
## Exercice 1 

### Reponse

**Enonce du probleme**

L'objectif est de predire si un demandeur de pret obtiendra une approbation ou non, ce qui correspond a une classification binaire ou la variable cible Loan_Status vaut Y (approuve) ou N (refuse). Cette prediction aide les institutions financieres a automatiser et a securiser leurs decisions d'octroi de credit.

**Types de donnees necessaires**

Premierement, les informations personnelles du demandeur comme le genre, la situation matrimoniale et le nombre de personnes a charge permettent d'evaluer la stabilite sociale et familiale de l'emprunteur. Deuxiemement, les informations professionnelles comme le niveau d'education et le statut d'employe ou de travailleur independant renseignent sur la regularite et la fiabilite du revenu. Troisiemement, les donnees financieres telles que le revenu du demandeur, le revenu du co-demandeur et le montant du pret demande sont directement liees a la capacite de remboursement. Quatriemement, les conditions du pret comme la duree et le montant permettent de calculer la charge mensuelle de remboursement. Enfin, l'historique de credit est la variable la plus predictive car elle indique si le demandeur a deja honore ses engagements financiers dans le passe.

**Sources de donnees**

La principale source est le systeme interne de la banque ou de l'institution de credit, qui contient les dossiers de demande et les resultats des prets passes. Les bureaux de credit fournissent les scores et les historiques de credit via des API. Des plateformes publiques comme Kaggle mettent a disposition des jeux de donnees similaires, comme le dataset Loan Prediction utilise dans cet exercice, pour le prototypage et l'entrainement des modeles.

**Risques et contraintes**

La collecte de ces donnees comporte plusieurs defis importants. Les lois sur la vie privee comme le RGPD exigent que les donnees personnelles soient traitees avec soin et que les demandeurs donnent leur consentement. Les lois anti-discrimination interdisent d'utiliser des attributs proteges comme le genre pour prendre des decisions de credit. Les donnees peuvent contenir des valeurs manquantes ou des erreurs qu'il faut corriger avant toute utilisation. Enfin, si les decisions de credit passees etaient biaisees envers certains groupes de demandeurs, le modele risque de reproduire ces biais.

## Exercice 2 

In [ ]:
import pandas as pd

# On charge le vrai jeu de donnees Loan Prediction
df = pd.read_csv('train_u6lujuX_CVtuZ9i (1).csv')

# On affiche les colonnes disponibles dans le dataset
print("Colonnes disponibles dans le dataset :")
print(df.columns.tolist())
print()

# On affiche un apercu des premieres lignes
print("Apercu des donnees :")
print(df.head())
print()

# On verifie les valeurs manquantes pour chaque colonne
print("Valeurs manquantes par colonne :")
print(df.isnull().sum())
print()

# On selectionne les variables les plus pertinentes pour predire l'approbation du pret
# On exclut Loan_ID car c'est un identifiant sans valeur predictive
# On exclut aussi Gender car c'est un attribut protege pouvant causer de la discrimination
variables_selectionnees = [
    "Credit_History",      # variable la plus predictive : historique de remboursement passe
    "ApplicantIncome",     # revenu du demandeur : capacite de remboursement directe
    "CoapplicantIncome",   # revenu du co-demandeur : capacite de remboursement complementaire
    "LoanAmount",          # montant demande : charge financiere a supporter
    "Loan_Amount_Term",    # duree du pret : impact sur les mensualites
    "Married",             # situation matrimoniale : stabilite financiere du foyer
    "Dependents",          # nombre de personnes a charge : charges du foyer
    "Education",           # niveau d'education : lien avec la stabilite professionnelle
    "Self_Employed",       # statut professionnel : regularite du revenu
    "Property_Area"        # zone geographique : reflete le niveau de vie et les risques locaux
]

# On affiche les variables selectionnees
print("Variables selectionnees pour le modele :")
for v in variables_selectionnees:
    print(" -", v)

# On affiche les statistiques de base sur les variables numeriques selectionnees
print()
print("Statistiques des variables numeriques :")
print(df[['ApplicantIncome','CoapplicantIncome','LoanAmount','Loan_Amount_Term','Credit_History']].describe())


### Justification

Les variables selectionnees a partir du jeu de donnees Loan Prediction sont : Credit_History, ApplicantIncome, CoapplicantIncome, LoanAmount, Loan_Amount_Term, Married, Dependents, Education, Self_Employed et Property_Area.

La variable Credit_History est de loin la plus importante car elle indique si le demandeur a deja rembourse ses dettes precedentes (valeur 1) ou non (valeur 0), ce qui est le signal le plus direct de sa fiabilite financiere. Le revenu du demandeur ApplicantIncome mesure sa capacite de remboursement mensuelle et un revenu plus eleve reduit le risque de defaut. Le revenu du co-demandeur CoapplicantIncome vient s'ajouter au revenu principal pour evaluer le revenu total du foyer, ce qui renforce la capacite de remboursement globale. Le montant du pret LoanAmount est important car un montant eleve par rapport au revenu augmente le risque que le demandeur ne puisse pas rembourser. La duree du pret Loan_Amount_Term influence le montant des mensualites et donc la charge mensuelle supportee par l'emprunteur. La situation matrimoniale Married peut indiquer une stabilite financiere plus grande car deux personnes partagent les charges du foyer. Le nombre de personnes a charge Dependents est pertinent car plus il est eleve, plus les charges fixes du menage sont importantes et moins il reste de revenu disponible pour rembourser le pret. Le niveau d'education Education est associe a de meilleures perspectives d'emploi et de revenu stable, ce qui reduit le risque de defaut. Le statut de travailleur independant Self_Employed est utile car les travailleurs independants ont un revenu plus variable que les salaries, ce qui peut influencer leur capacite de remboursement. Enfin, la zone geographique Property_Area reflete des differences de niveau de vie et de marche immobilier qui peuvent influer sur la capacite de remboursement selon que le demandeur habite en zone urbaine, semi-urbaine ou rurale.

La variable Loan_ID est exclue car c'est un simple identifiant unique qui ne contient aucune information predictive. La variable Gender est exclue car le genre est un attribut protege par les lois anti-discrimination et son utilisation pourrait conduire a des decisions de credit injustes.

Les variables categorielles comme Married, Education, Self_Employed, Dependents et Property_Area seront encodees avec le one-hot encoding, ce qui transforme chaque categorie en une colonne binaire separee que les algorithmes peuvent traiter. Les valeurs manquantes dans les colonnes numeriques comme LoanAmount, Loan_Amount_Term et Credit_History seront remplacees par la mediane de la colonne calculee uniquement sur l'ensemble d'entrainement, tandis que les valeurs manquantes dans les colonnes categorielles comme Gender, Married, Dependents et Self_Employed seront remplacees par la modalite la plus frequente de chaque colonne.

## Exercice 3 

In [ ]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, average_precision_score,
    confusion_matrix, classification_report
)

# On utilise des donnees fictives pour illustrer le calcul des metriques
# Dans un vrai projet, y_true contiendrait les vraies etiquettes du jeu de test
y_true = [0, 1, 0, 1, 0, 0, 1, 0, 1, 0]

# y_pred_proba contient les probabilites predites par le modele pour la classe 1 (approuve)
y_pred_proba = [0.05, 0.80, 0.10, 0.65, 0.20, 0.15, 0.70, 0.30, 0.85, 0.25]

# On fixe un seuil de decision : si la probabilite depasse ce seuil, on predit une approbation
seuil = 0.5
y_pred = [1 if p >= seuil else 0 for p in y_pred_proba]

# On calcule et affiche toutes les metriques d'evaluation
print("Exactitude (Accuracy) :", round(accuracy_score(y_true, y_pred), 4))
print("Precision :", round(precision_score(y_true, y_pred, zero_division=0), 4))
print("Rappel (Recall) :", round(recall_score(y_true, y_pred, zero_division=0), 4))
print("Score F1 :", round(f1_score(y_true, y_pred, zero_division=0), 4))
print("ROC-AUC :", round(roc_auc_score(y_true, y_pred_proba), 4))
print("PR-AUC (Precision moyenne) :", round(average_precision_score(y_true, y_pred_proba), 4))
print("\nMatrice de confusion :\n", confusion_matrix(y_true, y_pred))
print("\nRapport de classification :\n", classification_report(y_true, y_pred, zero_division=0))


### Reponse

**Choix du modele**

Pour ce probleme de prediction d'approbation de pret, un classificateur de type Foret Aleatoire (Random Forest) serait un bon modele principal car il gere bien les variables mixtes numeriques et categorielles, est robuste aux valeurs manquantes et fournit une mesure de l'importance des variables qui permet de mieux comprendre la decision. Un modele de regression logistique vaut egalement la peine d'etre entraine comme reference car il est simple, rapide a entrainer et ses resultats sont faciles a interpreter et a justifier aupres des decideurs.

**Etapes d'evaluation**

D'abord, le jeu de donnees est divise en un ensemble d'entrainement representant environ 80 pourcent des donnees et un ensemble de test representant 20 pourcent, en utilisant une division stratifiee pour conserver la meme proportion de prets approuves et refuses dans les deux ensembles. Ensuite le modele est entraine sur l'ensemble d'entrainement avec une validation croisee a 5 plis afin d'evaluer sa performance de maniere stable sans dependre d'une seule division. Le modele final est evalue sur l'ensemble de test qui n'a pas ete vu pendant l'entrainement pour obtenir une estimation non biaisee de ses performances reelles.

**Metriques**

Le dataset contient 422 prets approuves contre 192 refuses, ce qui cree un leger desequilibre de classes. L'exactitude seule n'est donc pas suffisante. On rapportera le score ROC-AUC comme metrique principale car il mesure la capacite du modele a distinguer les prets approuves des prets refuses sur tous les seuils possibles. Le rappel pour la classe de refus sera surveille attentivement car rater un mauvais dossier est plus couteux pour la banque qu'un refus injustifie. Le score F1 donnera une vue equilibree entre precision et rappel, et la matrice de confusion permettra de voir exactement combien d'erreurs de chaque type le modele commet.

**Desequilibre des classes**

Pour corriger le desequilibre entre prets approuves et refuses, on utilisera le parametre class_weight egal a balanced dans le modele, ce qui donne un poids plus important aux erreurs sur la classe minoritaire pendant l'entrainement.

**Seuil de decision**

Plutot que d'utiliser le seuil par defaut de 0.5, on analysera la courbe precision-rappel sur l'ensemble de validation pour choisir le seuil qui offre le meilleur equilibre entre l'approbation des bons dossiers et le rejet des mauvais dossiers selon les priorites metier de l'institution.

## Exercice 4 

### Reponse

**Scenario 1 - Prediction des cours boursiers**

Le type d'apprentissage automatique le plus adapte pour ce probleme est l'apprentissage supervise, plus precisement la regression. Les donnees d'entree sont les prix historiques, les volumes d'echange et des indicateurs financiers comme les moyennes mobiles. La sortie est une valeur numerique continue representant le prix futur predit de l'action. Le modele apprend en minimisant l'erreur entre ses predictions et les prix historiques reels, en respectant l'ordre temporel des observations.

**Scenario 2 - Organisation d'une bibliotheque de livres**

Le type d'apprentissage automatique le plus adapte pour ce probleme est l'apprentissage non supervise, plus precisement le clustering. Les donnees d'entree sont des representations textuelles de chaque livre comme son titre, sa description ou son contenu, converties en vecteurs numeriques. La sortie est un ensemble de groupes ou les livres au contenu ou aux themes similaires sont places ensemble. Comme il n'existe pas d'etiquettes de genre predefinies, le modele decouvre la structure par lui-meme en regroupant les livres qui se ressemblent.

**Scenario 3 - Navigation d'un robot dans un labyrinthe**

Le type d'apprentissage automatique le plus adapte pour ce probleme est l'apprentissage par renforcement. Les donnees d'entree sont la position actuelle du robot dans le labyrinthe ainsi que les actions disponibles comme se deplacer en haut, en bas, a gauche ou a droite. La sortie est une politique qui indique au robot quelle action effectuer dans chaque position. Le robot apprend en recevant une recompense positive quand il atteint la sortie et une petite penalite a chaque etape, de sorte qu'il apprend progressivement a trouver le chemin le plus court par essais et erreurs.

## Exercice 5 

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, average_precision_score

# On utilise des donnees fictives pour illustrer le calcul des metriques en classification supervisee
# Dans un vrai projet ces valeurs viendraient des predictions du modele sur le jeu de test
y_true = [0, 1, 1, 0, 1, 0, 0, 1, 0, 1]

# Probabilites predites par le modele pour la classe positive (pret approuve)
y_pred_proba = [0.1, 0.7, 0.8, 0.2, 0.6, 0.3, 0.4, 0.9, 0.2, 0.85]

# On convertit les probabilites en predictions binaires avec un seuil de 0.5
seuil = 0.5
y_pred = [1 if p >= seuil else 0 for p in y_pred_proba]

# Calcul et affichage des principales metriques de classification
print("Exactitude (Accuracy) :", round(accuracy_score(y_true, y_pred), 4))
print("Precision :", round(precision_score(y_true, y_pred, zero_division=0), 4))
print("Rappel (Recall) :", round(recall_score(y_true, y_pred, zero_division=0), 4))
print("Score F1 :", round(f1_score(y_true, y_pred, zero_division=0), 4))
print("ROC-AUC :", round(roc_auc_score(y_true, y_pred_proba), 4))
print("PR-AUC (Precision moyenne) :", round(average_precision_score(y_true, y_pred_proba), 4))


In [ ]:
from sklearn.datasets import make_blobs
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# On genere un jeu de donnees synthetique avec 3 groupes bien separes pour illustrer le clustering
X, _ = make_blobs(n_samples=300, centers=3, random_state=42)

# On applique l'algorithme K-Means avec 3 clusters
kmeans = KMeans(n_clusters=3, n_init="auto", random_state=42)
etiquettes = kmeans.fit_predict(X)

# On calcule le score de silhouette pour mesurer la qualite des clusters
# Un score proche de 1 signifie que les clusters sont bien separes et compacts
score_silhouette = silhouette_score(X, etiquettes)
print("Score de silhouette (plus c'est eleve, mieux c'est) :", round(score_silhouette, 4))

# Methode du coude : on trace l'inertie pour differentes valeurs de K
# L'inertie mesure la somme des distances entre chaque point et le centre de son cluster
# On choisit le K ou la courbe forme un coude, c'est-a-dire ou la diminution ralentit fortement
print("\nInertie pour differentes valeurs de K (methode du coude) :")
for k in range(2, 7):
    km = KMeans(n_clusters=k, n_init="auto", random_state=42)
    km.fit(X)
    print(f"  K={k} -> inertie = {round(km.inertia_, 2)}")


### Reponse

**Apprentissage supervise - Modele de classification (Random Forest)**

Pour evaluer un modele de classification supervise, on divise les donnees en un ensemble d'entrainement et un ensemble de test avec une division stratifiee. Le modele est entraine en utilisant une validation croisee a 5 plis afin que chaque partie des donnees soit utilisee pour la validation a un moment ou un autre. Sur l'ensemble de test, on rapporte l'exactitude, la precision, le rappel et le score F1 pour avoir une vision complete de la performance. On calcule aussi le score ROC-AUC pour mesurer la capacite du modele a separer les deux classes selon differents seuils, et on trace la courbe ROC pour visualiser ce resultat. La matrice de confusion est examinee pour comprendre combien de dossiers refusables ont ete approuves a tort et combien de bons dossiers ont ete refuses. Le principal defi est d'eviter la fuite de donnees, ce qui signifie que toute etape de preprocessing comme la normalisation ou l'imputation doit etre calculee uniquement sur les donnees d'entrainement puis appliquee aux donnees de test.

**Apprentissage non supervise - Modele de clustering (K-Means)**

Comme le clustering n'a pas d'etiquettes de reference, on utilise des metriques internes pour mesurer la qualite des clusters formes. Le score de silhouette est calcule pour chaque point et mesure a quel point il est similaire a son propre cluster par rapport aux autres clusters : un score proche de 1 signifie que les clusters sont bien separes, tandis qu'un score proche de 0 indique que les clusters se chevauchent. Pour choisir le nombre de clusters K, on utilise la methode du coude en tracant l'inertie pour differentes valeurs de K et en cherchant le point ou la courbe cesse de diminuer rapidement. On peut aussi visualiser les clusters en deux dimensions avec PCA ou t-SNE pour verifier visuellement si les groupes formes ont du sens. Le principal defi est qu'un bon score de silhouette ne garantit pas que les clusters correspondent a des categories significatives dans le monde reel, et une validation par un expert du domaine reste donc indispensable.

**Apprentissage par renforcement - Q-Learning pour la navigation dans un labyrinthe**

Pour evaluer un agent d'apprentissage par renforcement, on suit la recompense cumulee collectee a chaque episode et on la trace au fil de l'entrainement : si le modele apprend, cette courbe devrait augmenter et se stabiliser avec le temps. On suit aussi le nombre d'etapes que l'agent prend pour atteindre la sortie a chaque episode, et une diminution du nombre d'etapes confirme que l'agent trouve des chemins plus efficaces. L'equilibre entre exploration et exploitation est gere avec une strategie epsilon-greedy ou l'agent commence par explorer aleatoirement et reduit progressivement sa part d'exploration a mesure qu'il accumule de l'experience. A la fin de l'entrainement, l'agent est evalue en mode purement greedy avec epsilon egal a zero pour mesurer la politique finale. Le principal defi est qu'un agent bien entraine sur un labyrinthe fixe peut ne pas generaliser a de nouvelles configurations qu'il n'a jamais vues pendant l'entrainement.